# P&ID Medallion Pipeline — Concepts Walkthrough (Bronze → Silver)

This notebook illustrates, end to end, what we have built so far: a **Bronze**
(raw, immutable ingestion) → **Silver** (parse + topology reconstruction) pipeline
for P&ID interoperability exports (DEXPI/Proteus and INGR ISO-15926 PostProc),
on local Spark + Delta Lake.

It demonstrates the key concepts:

- **Bronze** stores the source XML *as-is* — content hash, format detection, project
  code and drawing revision captured, dedup on exact bytes.
- **Silver** *re-houses* the validated `pidtool`/`bppidsys` reconstruction (the
  "crown jewel") — recovering inline valves the raw graph lacks — and emits typed
  tables: components, segments, connections, equipment.
- The **oracle firewall**: the source turnover assignment is carried as *quarantined*
  lineage, never computed on.
- The **`flow_sense`** four-state directional overlay and the **`derived`** provenance
  flag on every reified connection.
- **Format parity**: DEXPI and PostProc flow through one code path into one schema.
- A real-data finding: **`seg_tag` is not unique** (the CDC anchor-collision risk).
- **Stage D**: a declarative expectation suite writes the **`silver_quality`**
  punch list; only two structural invariants hard-fail (fail for bugs, not data).

> **Run order matters.** After any kernel restart, run the cells top-to-bottom.
> Cell 1 *must* be first — it forces the venv's Spark 3.5.1 and blocks the system
> Spark 4 at `/opt/spark`.

## 0. Environment & pinned session

Two things bite on local WSL and are handled here:

1. **Which Spark.** A system `SPARK_HOME=/opt/spark` (Spark 4) shadows the venv's
   Spark 3.5.1 and breaks Delta (`DeltaCatalog` not found). Cell 1 strips it
   *before* `pyspark` is ever imported.
2. **One catalog, one warehouse.** The Hive metastore (`metastore_db/`) holds
   *names → locations*; the warehouse (`spark-warehouse/`) holds the *data*. We
   **pin both** to fixed paths so every session sees the same tables (embedded
   Derby is single-session — don't also run a `!python -m ...` subprocess while
   this notebook's session is live).

In [1]:
# --- CELL 1 — must run FIRST (before any `import pyspark`) ---
import os, sys
os.environ.pop("SPARK_HOME", None)                       # ignore system /opt/spark (Spark 4)
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if "/opt/spark" not in p)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]
assert "pyspark" not in sys.modules, "Restart the kernel and run THIS cell first."

from pathlib import Path

# repo_root = the ProjectData repo root (the folder containing bronze/ and silver/)
repo_root = Path.cwd()
while not (repo_root / "bronze").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
assert (repo_root / "bronze").is_dir(), f"Open this notebook inside the ProjectData repo (cwd={Path.cwd()})"

# --- the variables for this walkthrough ---
source_sample_dir = repo_root / "sample_data"                 # committed synthetic fixtures
source_dir        = repo_root / "data/exports/projectA"       # real Project A (DEXPI)
source_dir_B      = repo_root / "data/exports/projectB"       # real Project B (PostProc)
table_path        = repo_root / "_tmp" / "bronze_pid_documents"  # PATH-BASED Bronze (throwaway)
spark_warehouse   = repo_root / "spark-warehouse"             # managed-table data
metastore_db      = repo_root / "metastore_db"                # Hive/Derby catalog

print("repo_root       :", repo_root)
print("table_path      :", table_path)
print("spark_warehouse :", spark_warehouse)
print("metastore_db    :", metastore_db)

repo_root       : /home/dcamacho/dev/ProjectData
table_path      : /home/dcamacho/dev/ProjectData/_tmp/bronze_pid_documents
spark_warehouse : /home/dcamacho/dev/ProjectData/spark-warehouse
metastore_db    : /home/dcamacho/dev/ProjectData/metastore_db


In [2]:
# --- CELL 2 — build ONE pinned Delta+Hive session (venv Spark 3.5.1) ---
from bronze.spark_session import get_spark
spark = get_spark(extra_conf={
    "spark.sql.warehouse.dir": f"file:{spark_warehouse}",
    "spark.hadoop.javax.jdo.option.ConnectionURL":
        f"jdbc:derby:;databaseName={metastore_db};create=true",
})
import pyspark
from pyspark.sql import functions as F
print("pyspark :", pyspark.__file__)   # expect .../.venv/...  NOT /opt/spark
print("Spark   :", spark.version)      # expect 3.5.1
assert "/opt/spark" not in pyspark.__file__, "Still on system Spark 4 — restart kernel, run Cell 1 first."
assert spark.version.startswith("3.5"), f"Expected Spark 3.5.x, got {spark.version}"
print("OK — venv Spark 3.5.1, Delta + Hive ready.")

your 131072x1 screen size is bogus. expect trouble
26/09/02 18:13:55 WARN Utils: Your hostname, DC01NNCOL resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/02 18:13:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fb7851db-801a-4a4f-9ce8-71ba9c91f505;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 261ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

pyspark : /home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/__init__.py
Spark   : 3.5.1
OK — venv Spark 3.5.1, Delta + Hive ready.


In [3]:
# --- CELL 3 (optional) — clean slate for a reproducible demo ---
# Safe: _tmp Bronze is throwaway; Silver tables are rebuilt from Bronze below.
import shutil
shutil.rmtree(table_path, ignore_errors=True)
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    spark.sql(f"DROP TABLE IF EXISTS silver.{t}")
    shutil.rmtree(spark_warehouse / "silver.db" / t, ignore_errors=True)
print("clean slate ready")

26/09/02 18:14:04 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/02 18:14:04 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/02 18:14:07 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/09/02 18:14:07 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dcamacho@127.0.1.1
26/09/02 18:14:07 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


clean slate ready


## 1. Bronze — raw, immutable ingestion

Bronze lands each source file **verbatim**, one row per distinct byte-version, with:
`content` (raw bytes), a self-describing `content_hash` (`sha256:…`), the detected
`source_format` (DEXPI vs POSTPROC), the EPC `document_number` and derived
`project_code`, and the current `drawing_revision` / `drawing_revision_date`.
It **never interprets** the network model — that's Silver's job.

Here we ingest into a **path-based** Bronze table (`table_path`), which needs no
metastore at all.

In [4]:
# --- pick sources: prefer the real exports, fall back to the committed samples ---
def xmls(d): return sorted(Path(d).glob("*.xml")) if Path(d).is_dir() else []
sources = [d for d in (source_dir, source_dir_B) if xmls(d)]
if not sources:
    sources = [source_sample_dir]
for d in sources:
    print(f"{len(xmls(d)):3d} xml  in  {d}")

  5 xml  in  /home/dcamacho/dev/ProjectData/data/exports/projectB


In [5]:
# --- ingest each source folder into the SAME path-based Bronze table ---
from bronze.notebook import ingest_folder
for d in sources:
    summary = ingest_folder(spark, source_dir=str(d), table_path=str(table_path))
    print(d.name, "->", summary)

26/09/02 18:14:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


projectB -> {'ingest_run_id': '96e89bcd-34dc-4bf8-b791-0c424bd840ff', 'files_in_batch': 5, 'rows_inserted': 5, 'rows_skipped_already_present': 0}


In [6]:
# --- inspect Bronze: both formats, lineage columns, self-describing hash ---
bronze = spark.read.format("delta").load(str(table_path))
print("Bronze rows:", bronze.count())
bronze.groupBy("source_format").count().show()
bronze.select("document_number", "drawing_revision", "drawing_revision_date",
              "project_code", "content_hash").show(6, False)

Bronze rows: 5
+-------------+-----+
|source_format|count|
+-------------+-----+
|     POSTPROC|    5|
+-------------+-----+

+-----------------------------+----------------+---------------------+------------+-----------------------------------------------------------------------+
|document_number              |drawing_revision|drawing_revision_date|project_code|content_hash                                                           |
+-----------------------------+----------------+---------------------+------------+-----------------------------------------------------------------------+
|216097C-A14-PID-0021-0001-001|E               |2026/05/08           |216097C     |sha256:5aa863791cc76539ce823d57502777d1c6bb05fa48e80aaca233921540c70bfc|
|216097C-A14-PID-0021-0004-001|E               |2026/05/08           |216097C     |sha256:cab99a0b87087921133e67189b8afbce33fd9723ce34b7c1f89e76695fba8dd5|
|216097C-A14-PID-0021-0003-001|E               |2026/05/08           |216097C     |sha256:6888

**Dedup on exact bytes.** Re-ingesting the same files lands *nothing* new —
Bronze versions files by `content_hash`, so identical bytes are skipped
(`rows_skipped_already_present`).

In [7]:
# re-ingest the first folder — expect rows_inserted: 0
print(ingest_folder(spark, source_dir=str(sources[0]), table_path=str(table_path)))

{'ingest_run_id': '1c0cb934-2dfc-428a-8139-24023de09da3', 'files_in_batch': 5, 'rows_inserted': 0, 'rows_skipped_already_present': 5}


## 2. Silver — parse + topology reconstruction

Silver reads Bronze, picks the adapter from `source_format`, builds the DOM from
the raw bytes, and runs the **validated reconstruction** (vendored under
`silver/_recon/`, re-housed not re-derived). It emits four typed Delta tables and
carries the source turnover assignment as **quarantined** lineage.

We run it **in-session** (same notebook session) reading Bronze by path — so the
Silver tables land in this session's pinned catalog and are queryable by name.

In [8]:
# --- run Silver Stage A+B in-session ---
from silver.notebook import reconstruct
counts = reconstruct(spark, bronze_path=str(table_path), silver_schema="silver")
print(counts)

26/09/02 18:14:38 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_components` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/09/02 18:14:38 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/09/02 18:14:38 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/09/02 18:14:38 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/09/02 18:14:38 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/09/02 18:14:41 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`silver`.`silver_segments` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
2

{'silver_components': 1987, 'silver_segments': 807, 'silver_connections': 1396, 'silver_equipment': 9}


In [9]:
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    print(f"{t:22s} {spark.table('silver.' + t).count():6d} rows")

silver_components        1987 rows
silver_segments           807 rows
silver_connections       1396 rows
silver_equipment            9 rows


## 3. The concepts, illustrated in the data

### 3a. The crown jewel — inline valves recovered

The raw `<Connection>` records wire only each segment's two endpoints; inline valves
are missing. The reconstruction repairs the topology and re-inserts them. Here they
appear as real components flagged `is_valve` — in **both** formats.

In [10]:
spark.table("silver.silver_components") \
     .groupBy("source_format", "is_valve").count() \
     .orderBy("source_format", "is_valve").show()

+-------------+--------+-----+
|source_format|is_valve|count|
+-------------+--------+-----+
|     POSTPROC|   false| 1755|
|     POSTPROC|    true|  232|
+-------------+--------+-----+



### 3b. The oracle firewall

`src_turnover` / `src_subsystem` (the source commissioning assignment) is **carried**
on the segment row — but it sits on its own columns and **nothing computes on it**.
It is the validation *answer key*, quarantined so the ~97% agreement stays honest.

In [11]:
spark.table("silver.silver_segments") \
     .select("seg_tag", "fluid", "piping_materials_class",
             "src_turnover", "src_subsystem", "project_code").show(6, False)

+-------------------------+-----+----------------------+------------+-------------+------------+
|seg_tag                  |fluid|piping_materials_class|src_turnover|src_subsystem|project_code|
+-------------------------+-----+----------------------+------------+-------------+------------+
|3/4"-PG-1417205-D24P1HD-H|PG   |D24P1HD               |NULL        |NULL         |216097C     |
|4"-PG-1417205-D24P1HD-H  |PG   |D24P1HD               |NULL        |NULL         |216097C     |
|3/4"-PG-1417205-D24P1HD-H|PG   |D24P1HD               |NULL        |NULL         |216097C     |
|4"-PG-1417205-D24P1HD-H  |PG   |D24P1HD               |NULL        |NULL         |216097C     |
|4"-PG-1417205-D24P1HD-H  |PG   |D24P1HD               |NULL        |NULL         |216097C     |
|28"-PG-1417205-D24P1HD-H |PG   |D24P1HD               |NULL        |NULL         |216097C     |
+-------------------------+-----+----------------------+------------+-------------+------------+
only showing top 6 rows



### 3c. `flow_sense` — the four-state directional overlay

Direction is a *separate overlay* on the undirected connection, and it has four
states — `none` and `both` are real and a boolean couldn't hold them. Both formats
produce all four.

In [12]:
spark.table("silver.silver_connections") \
     .groupBy("source_format", "flow_sense").count() \
     .orderBy("source_format", "flow_sense").show()

+-------------+----------+-----+
|source_format|flow_sense|count|
+-------------+----------+-----+
|     POSTPROC|      both|    9|
|     POSTPROC|   forward|  616|
|     POSTPROC|      none|  135|
|     POSTPROC|   reverse|  636|
+-------------+----------+-----+



### 3d. `derived` — Source (stated) vs Derived (reconstructed) edges

Every reified connection carries provenance: `derived=false` where the edge was
stated in a source `<Connection>`, `derived=true` where the reconstruction inferred
it. This is what keeps the semantic layer from asserting inferred topology as fact.

In [13]:
spark.table("silver.silver_connections").groupBy("source_format", "derived").count().show()

+-------------+-------+-----+
|source_format|derived|count|
+-------------+-------+-----+
|     POSTPROC|   true|  726|
|     POSTPROC|  false|  670|
+-------------+-------+-----+



### 3e. Format parity — two standards, one schema

DEXPI and PostProc coexist in the same tables with identical columns — the
interoperability promise made concrete.

In [14]:
spark.table("silver.silver_segments").groupBy("source_format").count().show()

+-------------+-----+
|source_format|count|
+-------------+-----+
|     POSTPROC|  807|
+-------------+-----+



### 3f. Real-data finding — `seg_tag` is not unique

Distinct `segment_id`s can compose to the **same** business `seg_tag`. So the
composed tag cannot stand alone as the CDC segment anchor — it needs a
disambiguator, and the quality gate owes an *anchor-collision* flag. (This is why
we recorded it in the spec's §3.5.)

In [15]:
(spark.table("silver.silver_segments")
   .groupBy("seg_tag").count().filter("count > 1")
   .orderBy(F.desc("count")).show(10, False))

+------------------------+-----+
|seg_tag                 |count|
+------------------------+-----+
|Conn to process/supply- |37   |
|36"-PG-1417205-D24P1HD-H|24   |
|36"-PG-1418103-D341HD-H |22   |
|36"-PG-1415101-D341H-H  |15   |
|PG-1418103-D341HD-H     |14   |
|28"-PG-1416201-F341HD-H |12   |
|40"-PG-1416304-D341HD-H |12   |
|36"-PG-1417202-D24P1HD-H|12   |
|36"-PG-1415109-D341H-H  |10   |
|10"-WBF-2219107-H241S-H |9    |
+------------------------+-----+
only showing top 10 rows



## 3g. Stage D — the data-quality punch list

Stage D promotes the specs' advisory flags to a **declarative expectation suite**
(rules-as-data, `silver/quality_suite.py`) and writes `silver_quality` — the
per-drawing / per-project **punch list** a pre-commissioning engineer fixes at
source *before* systemization runs (segments missing fluid / piping-class /
diameter, tags that break the naming convention, the `seg_tag` anchor-collision,
prefix-integrity, orphans). The gate is **observe-and-record**: everything flags
and flows. Only two *structural invariants* — an **oracle leak** (§5) or an
**unflagged `derived` edge** (§4) — hard-fail, because those are pipeline bugs,
not dirty data.

It also denormalises a `quality_gate` enum (`clean`/`flagged`/`quarantined`) back
onto every object row, so a cautious consumer can filter without joining the
ledger.

In [16]:
# --- run Stage D in-session; it reads the four Silver tables ---
from silver.notebook import quality
# refdata_path lights up the reference-backed checks (unknown fluid/unit, naming);
# without it those skip cleanly. Point it at the project's Reference_Data.xlsx:
refdata_path = repo_root / "Reference_Data.xlsx"
summary = quality(spark, refdata_path=str(refdata_path) if refdata_path.exists() else None)
import json; print(json.dumps(summary, indent=2, default=str))

26/09/02 18:14:58 ERROR HiveAlterHandler: Failed to alter table silver.silver_quality
26/09/02 18:14:58 WARN HiveExternalCatalog: Could not alter schema of table `silver`.`silver_quality` in a Hive compatible way. Updating Hive metastore in Spark SQL specific format.
java.lang.reflect.InvocationTargetException
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at org.apache.spark.sql.hive.client.Shim_v2_1.alterTable(HiveShim.scala:1611)
	at org.apache.spark.sql.hive.client.HiveClientImpl.$anonfun$alterTableDataSchema$1(HiveClientImpl.scala:633)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.sql.hive.client.HiveClientImpl.$anonfun$withHiv

{
  "expectations_run": 14,
  "expectations_skipped": 0,
  "ledger_rows": 1285,
  "hard_failures": 0,
  "objects_flagged": 1463,
  "objects_quarantined": 0,
  "run_warnings": 0,
  "warnings": [],
  "skipped": [],
  "quality_table": "silver.silver_quality"
}


**The punch list** — one row per flag occurrence, ordered worst-first. This
is the artefact the engineer works from.

In [17]:
from pyspark.sql import functions as F
sev_rank = F.when(F.col("severity") == "error", 0).when(F.col("severity") == "warn", 1).otherwise(2)
(spark.table("silver.silver_quality")
   .withColumn("_r", sev_rank)
   .orderBy("_r", "flag")
   .select("severity", "gate", "object_kind", "flag", "drawing_number", "detail")
   .show(40, False))

+--------+----+-----------+---------------------------+-----------------------------+--------------------------------------------------------------------------------------------------+
|severity|gate|object_kind|flag                       |drawing_number               |detail                                                                                            |
+--------+----+-----------+---------------------------+-----------------------------+--------------------------------------------------------------------------------------------------+
|warn    |flag|component  |instrument_tag_noncompliant|216097C-A14-PID-0021-0001-001|instrument tag '-I-13.01' does not match the project instrument naming pattern                    |
|warn    |flag|component  |instrument_tag_noncompliant|216097C-A14-PID-0021-0001-001|instrument tag '-I-14.01' does not match the project instrument naming pattern                    |
|warn    |flag|component  |instrument_tag_noncompliant|216097C-A14-PID-0021

**Punch-list rollup by flag** — where the data gaps concentrate.

In [18]:
(spark.table("silver.silver_quality")
   .groupBy("flag", "severity", "gate").count()
   .orderBy(F.desc("count")).show(30, False))

+----------------------------+--------+----+-----+
|flag                        |severity|gate|count|
+----------------------------+--------+----+-----+
|orphan_component            |info    |flag|619  |
|segment_missing_diameter    |warn    |flag|157  |
|seg_tag_anchor_collision    |info    |flag|150  |
|segment_insulation_absent   |info    |flag|108  |
|segment_missing_piping_class|warn    |flag|104  |
|segment_missing_fluid       |warn    |flag|103  |
|instrument_tag_noncompliant |warn    |flag|33   |
|prefix_integrity            |warn    |flag|9    |
|segment_unknown_insulation  |warn    |flag|2    |
+----------------------------+--------+----+-----+



**The gate rollup on the objects themselves** — `flagged` rows still flow to
Gold and the rules; a strict consumer can exclude `quarantined` without a join.

In [19]:
for t in ["silver_segments", "silver_components"]:
    print(t)
    spark.table("silver." + t).groupBy("quality_gate").count().orderBy("quality_gate").show()

silver_segments
+------------+-----+
|quality_gate|count|
+------------+-----+
|       clean|  117|
|     flagged|  690|
+------------+-----+

silver_components
+------------+-----+
|quality_gate|count|
+------------+-----+
|       clean| 1358|
|     flagged|  629|
+------------+-----+



## 3h. Lineage trace — one attribute, Bronze bytes → Silver column

The whole point of carrying `bronze_id` / `content_hash` on every Silver row (§4)
is that any value traces back to the exact source bytes it came from. Here we
follow **insulation** end to end: the Silver `insul_purpose` column, the raw
`InsulPurpose` attribute pulled straight out of the Bronze XML, and the `seg_tag`
suffix (e.g. `-H`) are the *same source fact reached three ways*. Reading it from
the segment's own `<GenericAttributes>` block mirrors `pidsys.master_data.ga()`
exactly. The identical three-hop walk traces fluid, diameter, piping class, or the
quarantined oracle columns — insulation isn't special.

In [20]:
# --- Lineage trace: Insulation from Bronze bytes -> Silver columns ---
import xml.etree.ElementTree as ET

seg = spark.table("silver.silver_segments")

# 1) the Silver insulation columns + the lineage keys that trace each row to source
(seg.select("segment_id", "seg_tag", "insul_purpose", "insul_type", "insul_thick",
            "bronze_id", "content_hash", "drawing_number")
    .where("insul_purpose is not null")
    .show(8, False))

# 2) pull InsulPurpose straight out of the raw Bronze XML for one segment and compare.
#    Bronze is a named table in a full run; fall back to the path-based table (Cell 1).
try:
    bronze = spark.table("bronze.pid_documents")
except Exception:
    bronze = spark.read.format("delta").load(str(table_path))

row = (seg.where("insul_purpose is not null")
          .join(bronze.select("bronze_id", "content"), "bronze_id")
          .select("segment_id", "insul_purpose", "insul_type", "insul_thick", "content")
          .head())

def _ln(el):                                   # strip XML namespace
    return el.tag.split("}")[-1]

def insul_from_bytes(content, seg_id):
    # mirror pidsys.master_data.ga(): the segment's OWN <GenericAttributes> block
    root = ET.fromstring(bytes(content))
    for el in root.iter():
        if _ln(el) == "PipingNetworkSegment" and el.get("ID") == seg_id:
            return {g.get("Name"): g.get("Value")
                    for gas in el if _ln(gas) == "GenericAttributes"
                    for g in gas if _ln(g) == "GenericAttribute"
                    and (g.get("Name") or "").startswith("Insul")}
    return {}

if row is None:
    print("no segment with a non-null insul_purpose yet — run Silver Stage A+B first")
else:
    tag = seg.where(seg.segment_id == row.segment_id).head().seg_tag
    print("segment_id :", row.segment_id)
    print("seg_tag    :", tag, "  (last token = insulation purpose)")
    print("SILVER cols:", dict(insul_purpose=row.insul_purpose,
                               insul_type=row.insul_type, insul_thick=row.insul_thick))
    print("BRONZE XML :", insul_from_bytes(row.content, row.segment_id))
    # to trace a SPECIFIC flagged segment: replace the filter in `row` with
    #   .where("segment_id = '<the id from silver_quality.object_id>'")

+----------------------------------+------------------------+-------------+----------+-----------+------------------------------------+-----------------------------------------------------------------------+-----------------------------+
|segment_id                        |seg_tag                 |insul_purpose|insul_type|insul_thick|bronze_id                           |content_hash                                                           |drawing_number               |
+----------------------------------+------------------------+-------------+----------+-----------+------------------------------------+-----------------------------------------------------------------------+-----------------------------+
|SG622EA69EF79C416C943877B91E592C06|20"-SM-01-F242S-H       |H            |NULL      |NULL       |1d3a6237-b2fa-420d-a0d2-c2174298bf3a|sha256:5aa863791cc76539ce823d57502777d1c6bb05fa48e80aaca233921540c70bfc|216097C-A14-PID-0021-0001-001|
|SGF6DFB7F0FC7F4D17BBD6ED7D48CE3431|3/4"-WBF-221

26/09/02 18:15:11 WARN ObjectStore: Failed to get database bronze, returning NoSuchObjectException


segment_id : SG28149B027AA24BE2A37A0A98487B99D0
seg_tag    : SM-1404001-F242S-H   (last token = insulation purpose)
SILVER cols: {'insul_purpose': 'H', 'insul_type': None, 'insul_thick': None}
BRONZE XML : {'InsulPurpose': 'H'}


> If `insul_purpose` comes back all-null in Silver while the `seg_tag` still
> shows a `-H`/`-N` suffix, that mismatch *is* the finding — the value reached the
> composed tag but the column extraction missed it (Bronze→Silver drift), which is
> exactly what this trace is built to catch.

## 3i. Stage C — joining the P&IDs (off-page connectors)

Stage B reconstructs each drawing on its own; **Stage C** joins them into one
plant by matching **off-page connectors** (OPCs) across sheets — by `OPCTag` for
PostProc, by GUID for DEXPI (`bppidsys.offpage.match_pairs`, re-housed). Each
matched pair becomes one undirected, always-`derived` **`OffPage`** edge in
`silver_connections` that spans two drawings; an OPC whose mate isn't in the
loaded set is an **open boundary** — the system continues off-sheet — recorded as
an `opc_open_boundary` flag, never dropped. Run it after reconstruct().

In [22]:
# --- run Stage C in-session (harvest OPCs per sheet -> match across sheets) ---
from silver.notebook import assemble
import json
stats = assemble(spark, bronze_path=str(table_path))
print(json.dumps(stats, indent=2, default=str))

{
  "opc_records": 42,
  "opc_stitched": 6,
  "opc_offset": 30,
  "silver_connections": "silver.silver_connections"
}


**The cross-document edges** — one row per stitched OPC pair, `derived=true`,
joining two drawings into one connectivity graph.

In [23]:
off = spark.table("silver.silver_connections").where("conn_type = 'OffPage'")
print("OffPage edges:", off.count())
off.select("connection_id", "from_id", "to_id", "derived", "flow_sense").show(20, False)

OffPage edges: 6
+-----------------------------------------------------------------------+----------------------------------+----------------------------------+-------+----------+
|connection_id                                                          |from_id                           |to_id                             |derived|flow_sense|
+-----------------------------------------------------------------------+----------------------------------+----------------------------------+-------+----------+
|sha256:56659c0999da7abb4d6764b14674ea1e59d3fcfb19b6dcbe4f52b2b8196f7d82|SP5A970EF3697544E992295BEC0CC945D1|SP647F24E0C0AF4082A76710FD640AD236|true   |none      |
|sha256:8aa52e2ce3bd7ac959650189857381ddf4393cfdc50735e46dd69c0682bbd50c|SP1832754FE12D4AF6A4C010A658AF12FA|SP230325954370480EA1620C30DCDFA7F6|true   |none      |
|sha256:31ce937b8110469b29fef94073a9c1d6d1f3a5b0424682c0128060bccad934b4|SP48F67504784E42AB811FE4BC937E8518|SP7BF76D6845964F67A70844B976F6E63D|true   |none      |
|sha2

**Open boundaries** — OPCs with no mate in the loaded set. Not errors: the
system continues onto a sheet that wasn't loaded. Load more sheets and these
resolve into `OffPage` edges.

In [24]:
(spark.table("silver.silver_quality").where("flag = 'opc_open_boundary'")
   .select("object_id", "drawing_number", "detail").show(20, False))

+----------------------------------+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|object_id                         |drawing_number               |detail                                                                                                                                                                      |
+----------------------------------+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|SP55813A8E259A4A7F837BE3D6EC1E6B02|216097C-A14-PID-0021-0005-001|OPC '4863' on drawing 216097C-A14-PID-0021-0005-001 (paired drawing A22-0051-003) has no mate in the loaded set — open boundary (the system continues off-set; not an error)|
|SP2E3DA1DA4BDB4EA48C1D1EEC58B3BCE4|2160

## 4. Recap

**Built (Phase-1 + Stage C + Stage D):** Bronze (raw, immutable, dedup,
format-tagged) → Silver (parse + reconstruction, four typed tables) → **Stage C
assembly** (`OffPage` edges + open boundaries) → **Stage D quality gate**
(`silver_quality` punch list + `quality_gate` rollup), validated on real Project A
**and** Project B.

**Concepts shown:** store-as-is + content hash; format detection; the reconstruction
recovering inline valves; the oracle firewall; the `flow_sense` enum and `derived`
provenance; format parity; the `seg_tag` anchor-collision; and the Stage-D
punch list with its fail-for-bugs-not-data gate policy.

**Runtime lessons baked in:** force the venv's Spark 3.5.1 (Cell 1); pin the metastore
+ warehouse; run in-session; Bronze can be path-based to sidestep the catalog entirely.

**Next:** Stage E (object-grain CDC), then the Gold layer (bi-temporal + RDF/IDO).
Stage D's reference-backed checks (unknown fluid/unit, insulation, equipment/
instrument naming) light up as soon as a project `Reference_Data.xlsx` — with
`Naming` and `Insulation` sheets — is supplied.